# Paper Figures


Final figure generation for the ICML 2026 workshop submission.


In [ ]:
from functools import cache
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import wandb
from matplotlib.lines import Line2D
from wandb.apis.public.runs import Run

PROJECT = "graphcore/llama-mobile"
FIG_PATH = Path("/Users/lukar/Code/llama-mobile/figs")

DOWNSTREAM_TASKS = ["vqa", "chartqa", "docvqa", "ai2d"]
TASK_TITLES = {
    "vqa": "VQAv2",
    "chartqa": "ChartQA",
    "docvqa": "DocVQA",
    "ai2d": "AI2D",
}


# Global plot parameters
COLORS = {
    "int_qat": "#4C78A8",
    "int_direct": "#8FB1D4",
    "student_t": "#54A24B",
    "lloyd_max": "#F58518",
    "s3d8": "#E45756",
    "bf16": "0.35",
}
POINT_SIZE = 60
DIRECT_POINT_SIZE = 70
STAR_SIZE = 180
POINT_ALPHA = 0.95
EDGE_WIDTH = 0.6
LINE_WIDTH = 1.1
GRID_STYLE = {"axis": "y", "linestyle": ":", "linewidth": 0.6, "alpha": 0.45}

LABELS = {
    "int": r"$\mathtt{INT}$",
    "int_qat": r"$\mathtt{INT}$ + QAT",
    "int_direct": r"$\mathtt{INT}$ + direct cast",
    "int_gptq": r"$\mathtt{INT}$ + GPTQ",
    "bf16": r"$\mathtt{bfloat16}$ baseline",
    "s3d8": r"$\mathtt{S3D8}$",
}
FORMAT_PLOT_LABELS = {
    "INT": r"$\mathtt{INT}$",
    "student-t": r"$\mathtt{student{-}t}$",
    "lloyd-max": r"$\mathtt{lloyd{-}max}$",
    "S3D8": r"$\mathtt{S3D8}$",
}
FORMAT_LATEX_LABELS = {
    "INT": r"\texttt{INT}",
    "student-t": r"\texttt{student-t}",
    "lloyd-max": r"\texttt{lloyd-max}",
    "S3D8": r"\texttt{S3D8}",
}

sns.set_theme(style="ticks", context="paper")
plt.rcParams.update(
    {
        "font.family": "STIXGeneral",
        "mathtext.fontset": "stix",
        "font.size": 10,
        "axes.labelsize": 11,
        "axes.titlesize": 12,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "legend.fontsize": 9,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    }
)


In [ ]:
# Set to True to save pdf figures to FIG_PATH
SAVE_FIGS = False

In [ ]:
api = wandb.Api()


In [ ]:
# Data helpers
def get_metric(task: str) -> str:
    from eval.vqa import TASKS

    metrics = TASKS[task].METRICS
    assert len(metrics) == 1
    return metrics[0]


def overall_metric(scores: list[float]) -> float:
    return sum(scores) / len(scores)


def get_field(x: Any, key: str) -> Any:
    if isinstance(x, dict):
        return x.get(key)
    return getattr(x, key, None)


def activation_fmt_name(fmt: Any) -> str:
    if fmt is None:
        return "bf16"

    fmt_type = get_field(fmt, "_type")
    element_format = get_field(fmt, "element_format")
    if fmt_type == "int" or (
        fmt_type == "linear" and get_field(element_format, "_type") == "int"
    ):
        return "int"
    return str(fmt)


@cache
def get_num_params(model_name: str) -> int:
    from accelerate import init_empty_weights
    from transformers import AutoConfig, MllamaForConditionalGeneration

    config = AutoConfig.from_pretrained(model_name)
    with init_empty_weights():
        model = MllamaForConditionalGeneration(config)
    return sum(p.numel() for p in model.parameters())


def get_prompt_type(train_data: list[dict[str, Any]]) -> str:
    assert len(train_data) == 1
    prompt_path = train_data[0]["path"].split("/")[-1]
    if prompt_path == "single-prompt-1280k":
        return "fixed"
    if prompt_path == "new-prompts-1280k":
        return "sampled"
    raise ValueError(f"unexpected prompt path: {prompt_path}")


def quantisation_format_name(fmt: dict[str, Any]) -> str:
    fmt_type = fmt["_type"]
    if fmt_type == "linear":
        element_format = fmt["element_format"]
        if element_format["_type"] == "lut":
            df = int(element_format["name"].split("[")[1].split("]")[0])
            return f"student-t [df={df}]"
        return element_format["_type"]

    if fmt_type == "fit_scaled":
        if fmt["element_family"] == "t":
            return "student-t [fit]"
        if fmt["element_family"] == "lloyd_max":
            return "lloyd-max [fit]"
        if fmt["element_family"] == "s3d8":
            return "s3d8"

    raise ValueError(f"unexpected format type: {fmt_type}")


def cleanup_run(run: Run) -> dict[str, Any]:
    config = run.config
    scores = {
        task: run.summary[f"downstream.{task}.{get_metric(task)}"]
        for task in DOWNSTREAM_TASKS
    }
    history = run.history()

    return {
        "name": run.name,
        "size": config["n_bits"] / (8 * 10**6),
        "bits_per_param": config["n_bits"] / get_num_params(config["model_name"]),
        "n_steps": config["training"]["n_steps"],
        "lr": config["training"]["optimiser"]["lr"],
        "prompt": get_prompt_type(config["data"]["train"]),
        "rotation": config["training"].get("rotate_text_residual"),
        "activation_fmt": activation_fmt_name(config["quantisation"].get("activation_fmt")),
        "fmt": quantisation_format_name(config["quantisation"]["fmt"]),
        "overall": overall_metric(list(scores.values())),
        "train_loss": history["train/loss"].tail(10).mean()
        if "train/loss" in history
        else None,
        "val_loss": run.summary.get("val/loss"),
        **scores,
    }


def load_runs(wandb_api: Any, config_name: str):
    return wandb_api.runs(
        PROJECT, filters={"config.name": config_name, "state": "finished"}
    )


def runs_dataframe(wandb_api: Any, config_name: str) -> pd.DataFrame:
    return (
        pd.DataFrame([cleanup_run(run) for run in load_runs(wandb_api, config_name)])
        .sort_values(["size", "overall"])
        .reset_index(drop=True)
    )


def labeled_runs_dataframe(
    wandb_api: Any,
    run_names: dict[str, str],
    label_col: str,
    *,
    n_steps: int | None = None,
) -> pd.DataFrame:
    frames = []
    for label, run_name in run_names.items():
        df = runs_dataframe(wandb_api, run_name)
        if n_steps is not None:
            df = df.loc[df["n_steps"] == n_steps]
        frames.append(df.assign(**{label_col: label}))
    return pd.concat(frames, ignore_index=True)


In [ ]:
# Plot helpers
def finish_axis(ax, *, x_margin: float = 0.04, y_margin: float = 0.08) -> None:
    ax.grid(**GRID_STYLE)
    ax.margins(x=x_margin, y=y_margin)
    sns.despine(ax=ax)


def add_bits_axis(ax, size_to_bpp: float):
    secax = ax.secondary_xaxis(
        "top",
        functions=(lambda size: size * size_to_bpp, lambda bpp: bpp / size_to_bpp),
    )
    secax.set_xlabel("Bits per parameter", color="0.4", labelpad=5)
    secax.tick_params(axis="x", colors="0.45", labelsize=8, pad=2)
    secax.spines["top"].set_color("0.6")
    secax.spines["top"].set_alpha(0.5)
    return secax


def finish_size_axis(ax, size_to_bpp: float, **kwargs: Any) -> None:
    finish_axis(ax, **kwargs)
    add_bits_axis(ax, size_to_bpp)


def style_label(key: str, style: dict[str, Any]) -> str:
    return style.get("label", key)


def scatter_styled(
    ax,
    df: pd.DataFrame,
    x: str,
    y: str,
    style: dict[str, Any],
    *,
    label: str | None = None,
) -> None:
    ax.scatter(
        df[x],
        df[y],
        s=style.get("size", POINT_SIZE),
        marker=style.get("marker", "o"),
        color=style["color"],
        edgecolor=style.get("edgecolor", "white"),
        linewidth=style.get("linewidth", EDGE_WIDTH),
        alpha=POINT_ALPHA,
        zorder=style.get("zorder", 3),
        label=label,
    )


def line_styled(
    ax,
    df: pd.DataFrame,
    x: str,
    y: str,
    style: dict[str, Any],
    *,
    label: str | None = None,
) -> None:
    ax.plot(
        df[x],
        df[y],
        marker=style.get("marker", "o"),
        markersize=6,
        color=style["color"],
        linewidth=LINE_WIDTH,
        label=label,
    )


def scatter_groups(
    ax,
    df: pd.DataFrame,
    group_col: str,
    styles: dict[str, dict[str, Any]],
    x: str,
    y: str,
    *,
    label: bool = False,
) -> None:
    for key, style in styles.items():
        group_df = df.loc[df[group_col] == key]
        if not group_df.empty:
            scatter_styled(
                ax,
                group_df,
                x,
                y,
                style,
                label=style_label(key, style) if label else None,
            )


def line_groups(
    ax,
    df: pd.DataFrame,
    group_col: str,
    styles: dict[str, dict[str, Any]],
    x: str,
    y: str,
    *,
    label: bool = False,
) -> None:
    for key, style in styles.items():
        group_df = df.loc[df[group_col] == key]
        if not group_df.empty:
            line_styled(
                ax,
                group_df,
                x,
                y,
                style,
                label=style_label(key, style) if label else None,
            )


def plot_int_runs(ax, df: pd.DataFrame, y: str, *, direct: bool = False) -> None:
    if df.empty:
        return

    if direct:
        ax.scatter(
            df["size"],
            df[y],
            s=DIRECT_POINT_SIZE,
            marker="x",
            color=COLORS["int_direct"],
            linewidth=1.2,
            alpha=POINT_ALPHA,
            zorder=4,
        )
    else:
        scatter_styled(ax, df, "size", y, {"color": COLORS["int_qat"]})


def plot_gptq_runs(ax, df: pd.DataFrame, y: str) -> None:
    scatter_styled(
        ax,
        df,
        "size",
        y,
        {
            "color": "none",
            "marker": "D",
            "edgecolor": COLORS["int_qat"],
            "linewidth": 1.2,
        },
    )


def plot_bf16(ax, y: float) -> None:
    ax.axhline(y, linestyle="--", linewidth=LINE_WIDTH, color=COLORS["bf16"])


def plot_s3d8(ax, x: float, y: float) -> None:
    ax.scatter(
        [x],
        [y],
        marker="*",
        s=STAR_SIZE,
        color=COLORS["s3d8"],
        edgecolor="black",
        linewidth=0.8,
        zorder=10,
    )


def style_handles(styles: dict[str, dict[str, Any]]) -> list[Line2D]:
    return [
        Line2D(
            [],
            [],
            marker=style.get("marker", "o"),
            linestyle="",
            markersize=6,
            markerfacecolor=style["color"],
            markeredgecolor=style.get("edgecolor", "white"),
            markeredgewidth=EDGE_WIDTH,
            label=style_label(key, style),
        )
        for key, style in styles.items()
    ]


def s3d8_handle() -> Line2D:
    return Line2D(
        [],
        [],
        marker="*",
        linestyle="",
        markersize=10,
        color=COLORS["s3d8"],
        markeredgecolor="black",
        label=LABELS["s3d8"],
    )


def gptq_handle() -> Line2D:
    return Line2D(
        [],
        [],
        marker="D",
        linestyle="",
        markersize=6,
        markerfacecolor="none",
        markeredgecolor=COLORS["int_qat"],
        markeredgewidth=1.2,
        label=LABELS["int_gptq"],
    )


def model_legend_handles(*, compact: bool = False) -> list[Line2D]:
    handles = [
        Line2D(
            [],
            [],
            marker="o",
            linestyle="",
            markersize=6,
            markerfacecolor=COLORS["int_qat"],
            markeredgecolor="white",
            markeredgewidth=EDGE_WIDTH,
            label=LABELS["int_qat"],
        ),
        Line2D(
            [],
            [],
            marker="x",
            linestyle="",
            markersize=7,
            color=COLORS["int_direct"],
            markeredgewidth=1.2,
            label=LABELS["int_direct"],
        ),
        s3d8_handle(),
        Line2D(
            [],
            [],
            linestyle="--",
            linewidth=LINE_WIDTH,
            color=COLORS["bf16"],
            label=LABELS["bf16"],
        ),
    ]
    if compact:
        return handles
    return [
        Line2D(
            [],
            [],
            marker="x",
            linestyle="",
            markersize=7,
            color=COLORS["int_direct"],
            markeredgewidth=1.2,
            label=LABELS["int_direct"],
        ),
        Line2D(
            [],
            [],
            marker="o",
            linestyle="",
            markersize=6,
            markerfacecolor=COLORS["int_qat"],
            markeredgecolor="white",
            markeredgewidth=EDGE_WIDTH,
            label=LABELS["int_qat"],
        ),
        handles[-1],
    ]


def save_figure(fig, filename: str) -> None:
    if SAVE_FIGS:
        fig.savefig(FIG_PATH / filename, bbox_inches="tight")


### Main Figure

Compare $\mathtt{INT}$ block-scaled baselines against the selected $\mathtt{S3D8}$ run. Baseline points use $\mathtt{INT}$ activations, no rotations, and direct casting, 2k-step QAT, or GPTQ.

Runs:
- $\mathtt{INT}$ baseline: `int-baseline-22-04-26`
- $\mathtt{INT}$ + GPTQ: `data/gptq-results.csv` (`INT absmax` rows; raw bpp is adjusted below to use whole-model accounting)
- $\mathtt{S3D8}$: `new-fmt-21-04-26`
- bfloat16 reference: `whole-fire-1607`

In [ ]:
int_baseline_run = "int-baseline-22-04-26"
s3d8_run = "new-fmt-21-04-26"

baseline_df = runs_dataframe(api, int_baseline_run)
df_new = runs_dataframe(api, s3d8_run)

display(baseline_df.sort_values("size"))
display(df_new.sort_values(["size", "overall"]))


In [ ]:
best_s3d8_run = "proud-sponge-1878"

best_run = df_new.loc[df_new["name"] == best_s3d8_run].iloc[0]
display(best_run)


In [ ]:
bf16_run_name = "whole-fire-1607"

bf16_run = api.runs(
    PROJECT,
    filters={"display_name": bf16_run_name, "state": "finished"},
)[0]

baseline = {
    task: bf16_run.summary[f"downstream.{task}.{get_metric(task)}"]
    for task in DOWNSTREAM_TASKS
}
baseline["overall"] = overall_metric(list(baseline.values()))
baseline_size = bf16_run.config["n_bits"] / (8 * 10**6)

display(baseline)
print(f"Original size: {baseline_size:.0f} MB")


In [ ]:
qat_df = baseline_df.loc[baseline_df["n_steps"] == 2048]
direct_cast_df = baseline_df.loc[baseline_df["n_steps"] == 0]
other_int_df = baseline_df.loc[~baseline_df["n_steps"].isin([0, 2048])]

# Main Figure exclusions as (# codepoints, group size). All available INT
# methods at each listed x-position are omitted. Use set() to show all.
MAIN_FIGURE_EXCLUDED_INT_CONFIGS = {(8, 128)}

our_size = best_run["size"]
our_perf = best_run["overall"]
our_task_scores = best_run[DOWNSTREAM_TASKS].to_dict()
our_loss_scores = best_run[["train_loss", "val_loss"]].to_dict()
size_to_bpp = baseline_df["bits_per_param"].iloc[0] / baseline_df["size"].iloc[0]

gptq_df = pd.read_csv("data/gptq-results.csv")
gptq_df = gptq_df.loc[gptq_df["Format"] == "INT absmax"].copy()

# The CSV bpp covers quantised weights and scales. Match the QAT/direct-cast
# whole-model accounting by including unquantised parameters at bfloat16.
n_quantised_params = 10_666_613_760
n_unquantised_params = 3_607_075
unquantised_param_bits = 16
n_model_params = n_quantised_params + n_unquantised_params
gptq_df["raw_bpp"] = gptq_df["bpp"]
gptq_df["bpp"] = (
    gptq_df["raw_bpp"] * n_quantised_params
    + unquantised_param_bits * n_unquantised_params
) / n_model_params

gptq_df["size"] = gptq_df["bpp"] / size_to_bpp
gptq_df["overall"] = gptq_df["average_accuracy"]

gptq_configs = pd.MultiIndex.from_frame(
    gptq_df[["#codepoints", "group-size"]]
)
excluded_gptq_mask = gptq_configs.isin(MAIN_FIGURE_EXCLUDED_INT_CONFIGS)
excluded_int_sizes = gptq_df.loc[excluded_gptq_mask, "size"].to_numpy()


def exclude_main_figure_int_sizes(df: pd.DataFrame) -> pd.DataFrame:
    if not excluded_int_sizes.size:
        return df
    excluded_mask = np.isclose(
        df["size"].to_numpy()[:, None],
        excluded_int_sizes,
        rtol=0,
        atol=1e-6,
    ).any(axis=1)
    return df.loc[~excluded_mask]


main_figure_qat_df = exclude_main_figure_int_sizes(qat_df)
main_figure_direct_cast_df = exclude_main_figure_int_sizes(direct_cast_df)
main_figure_other_int_df = exclude_main_figure_int_sizes(other_int_df)
main_figure_gptq_df = gptq_df.loc[~excluded_gptq_mask]
display(gptq_df.sort_values("bpp"))


In [ ]:
figsize = (5.2, 3.6)
fig, ax = plt.subplots(figsize=figsize, constrained_layout=True)

for df, direct in [
    (main_figure_qat_df, False),
    (main_figure_other_int_df, False),
    (main_figure_direct_cast_df, True),
]:
    plot_int_runs(ax, df, "overall", direct=direct)
plot_gptq_runs(ax, main_figure_gptq_df, "overall")
plot_bf16(ax, baseline["overall"])
plot_s3d8(ax, our_size, our_perf)

ax.annotate(
    LABELS["s3d8"],
    xy=(our_size, our_perf),
    xytext=(10, -10),
    textcoords="offset points",
    fontsize=10,
    ha="left",
    va="bottom",
)
ax.set(
    # title="Overall Downstream Performance",
    xlabel="Model size [MB]",
    ylabel="Average task performance",
)
finish_size_axis(ax, size_to_bpp)

legend_handles = model_legend_handles()
legend_handles.insert(1, gptq_handle())
ax.legend(
    handles=legend_handles,
    loc="lower right",
    frameon=True,
    fancybox=False,
    edgecolor="0.8",
    labelspacing=0.35,
    handletextpad=0.5,
)

save_figure(fig, "fig1.pdf")
plt.show()


### Downstream Tasks


In [ ]:
figsize = (6.8, 5.6)
fig, axes = plt.subplots(2, 2, figsize=figsize, sharex=True, constrained_layout=True)

for ax, task in zip(axes.flatten(), DOWNSTREAM_TASKS):
    for df, direct in [(qat_df, False), (other_int_df, False), (direct_cast_df, True)]:
        plot_int_runs(ax, df, task, direct=direct)
    plot_bf16(ax, baseline[task])
    plot_s3d8(ax, our_size, our_task_scores[task])

    ax.set(title=TASK_TITLES[task], xlabel="Model size [MB]", ylabel=get_metric(task))
    finish_size_axis(ax, size_to_bpp)

fig.legend(
    handles=model_legend_handles(compact=True),
    loc="lower center",
    ncol=4,
    mode="expand",
    frameon=False,
    bbox_to_anchor=(0.0, -0.09, 1.0, 0.08),
    labelspacing=0.35,
    handletextpad=0.5,
)

save_figure(fig, "fig_tasks.pdf")
plt.show()


### Loss Curves


In [ ]:
loss_specs = [("train_loss", "Training loss"), ("val_loss", "Validation loss")]
figsize = (6.8, 3.0)
fig, axes = plt.subplots(1, 2, figsize=figsize, constrained_layout=True)

for ax, (key, title) in zip(axes, loss_specs):
    plot_int_runs(ax, baseline_df.dropna(subset=[key]), key)
    plot_s3d8(ax, our_size, our_loss_scores[key])

    ax.set(title=title, xlabel="Model size [MB]", ylabel="KL Loss")
    finish_size_axis(ax, size_to_bpp, y_margin=0.10)

fig.legend(
    handles=model_legend_handles(compact=True)[::2],
    loc="lower center",
    ncol=2,
    frameon=False,
    bbox_to_anchor=(0.5, -0.08),
)

# save_figure(fig, "fig_loss.pdf")
plt.show()


### Prompt Sampling vs. Fixed Prompt


Compare sampled prompts against a fixed `Describe this image` prompt. Both settings use the same 75% instruction-template mix, $\mathtt{INT3}$ weights with 64-value block scales, and steps in `[256, 512, 1024, 2048]`.

Run: `prompt-comparison-24-04-26`


In [ ]:
prompt_run = "prompt-comparison-24-04-26"
max_num_steps = 4096
prompt_styles = {
    "sampled": {"color": COLORS["int_qat"], "marker": "o", "label": "Sampled"},
    "fixed": {"color": COLORS["s3d8"], "marker": "s", "label": "Fixed"},
}

prompt_df = runs_dataframe(api, prompt_run)
plot_prompt_df = prompt_df.loc[prompt_df["n_steps"] <= max_num_steps]
prompt_steps = sorted(plot_prompt_df["n_steps"].unique())

display(prompt_df)


In [ ]:
figsize = (5.2, 3.2)
fig, ax = plt.subplots(figsize=figsize, constrained_layout=True)

scatter_groups(ax, plot_prompt_df, "prompt", prompt_styles, "n_steps", "overall", label=True)
ax.set(
    # title="Prompt Sampling vs. Fixed Prompt",
    xlabel="Training steps",
    ylabel="Average task performance",
    xticks=prompt_steps,
)
ax.set_ylim(bottom=0)
finish_axis(ax, x_margin=0.08)

ax.legend(
    title="Prompt type",
    loc="lower right",
    frameon=True,
    fancybox=False,
    edgecolor="0.8",
    labelspacing=0.35,
    handletextpad=0.5,
)

save_figure(fig, "fig_prompt_overall.pdf")
plt.show()


In [ ]:
figsize = (6.8, 5.6)
fig, axes = plt.subplots(2, 2, figsize=figsize, sharex=True, constrained_layout=True)

for ax, task in zip(axes.flatten(), DOWNSTREAM_TASKS):
    scatter_groups(ax, plot_prompt_df, "prompt", prompt_styles, "n_steps", task)
    ax.set(
        title=TASK_TITLES[task],
        xlabel="Training steps",
        ylabel=get_metric(task),
        xticks=prompt_steps,
    )
    ax.set_ylim(bottom=0)
    finish_axis(ax, x_margin=0.08)
    ax.tick_params(axis="x", labelrotation=45)
    for label in ax.get_xticklabels():
        label.set_ha("right")

fig.legend(
    handles=style_handles(prompt_styles),
    title="Prompt type",
    loc="lower center",
    ncol=2,
    frameon=False,
    bbox_to_anchor=(0.5, -0.08),
    columnspacing=1.2,
    labelspacing=0.35,
    handletextpad=0.5,
)

save_figure(fig, "fig_prompt_tasks.pdf")
plt.show()


### Learning Rate Sweep


Plot 2048-step runs from `lr-sweep-17-03-26` and `lr-sweep-27-04-26` against optimiser learning rate.


In [ ]:
lr_sweep_runs = {
    "without activation quantisation": "lr-sweep-17-03-26",
    "with activation quantisation": "lr-sweep-27-04-26",
}
lr_styles = {
    "without activation quantisation": {"color": COLORS["int_direct"], "marker": "s"},
    "with activation quantisation": {"color": COLORS["int_qat"], "marker": "o"},
}

lr_df = labeled_runs_dataframe(api, lr_sweep_runs, "activation_quantisation", n_steps=2048)
lr_df = lr_df.sort_values(["activation_quantisation", "lr"])
lr_ticks = sorted(lr_df["lr"].unique())
lr_tick_labels = [rf"$2^{{{round(np.log2(lr))}}}$" for lr in lr_ticks]

display(lr_df[["activation_quantisation", "name", "lr", "overall", "train_loss", "val_loss"]])


In [ ]:
figsize = (5.2, 3.2)
fig, ax = plt.subplots(figsize=figsize, constrained_layout=True)

line_groups(ax, lr_df, "activation_quantisation", lr_styles, "lr", "overall", label=True)
ax.set_xscale("log", base=2)
ax.set(
    # title="Learning Rate Sweep",
    xlabel="Learning rate",
    ylabel="Average task performance",
    xticks=lr_ticks,
    xticklabels=lr_tick_labels,
)
finish_axis(ax, x_margin=0.08)
ax.legend(frameon=True, fancybox=False, edgecolor="0.8")

# save_figure(fig, "fig_lr_overall.pdf")
plt.show()


In [ ]:
loss_specs = [("train_loss", "Training loss"), ("val_loss", "Validation loss")]
figsize = (6.8, 3.0)
fig, axes = plt.subplots(1, 2, figsize=figsize, constrained_layout=True)

for ax, (key, title) in zip(axes, loss_specs):
    line_groups(ax, lr_df, "activation_quantisation", lr_styles, "lr", key, label=True)
    ax.set_xscale("log", base=2)
    ax.set(
        title=title,
        xlabel="Learning rate",
        ylabel="KL Loss",
        xticks=lr_ticks,
        xticklabels=lr_tick_labels,
    )
    finish_axis(ax, x_margin=0.08, y_margin=0.10)

axes[0].legend(frameon=True, fancybox=False, edgecolor="0.8")

# save_figure(fig, "fig_lr_loss.pdf")
plt.show()


### Format Comparison


Compare 2048-step QAT points from $\mathtt{INT}$, $\mathtt{student{-}t}$, and scalar $\mathtt{lloyd{-}max}$ baselines against the selected $\mathtt{S3D8}$ run.


In [ ]:
format_runs = {
    "INT": "int-baseline-22-04-26",
    "student-t": "student-t-baseline-22-04-26",
    "lloyd-max": "lloyd-max-baseline-22-04-26",
}
format_styles = {
    "INT": {"color": COLORS["int_qat"], "marker": "o", "label": FORMAT_PLOT_LABELS["INT"]},
    "student-t": {"color": COLORS["student_t"], "marker": "D", "label": FORMAT_PLOT_LABELS["student-t"]},
    "lloyd-max": {"color": COLORS["lloyd_max"], "marker": "^", "label": FORMAT_PLOT_LABELS["lloyd-max"]},
}

format_df = labeled_runs_dataframe(api, format_runs, "format", n_steps=2048)
format_df = format_df.sort_values(["format", "size", "overall"])

format_display_df = format_df.assign(format=format_df["format"].replace(FORMAT_LATEX_LABELS))
display(format_display_df[["format", "name", "size", "bits_per_param", "overall", *DOWNSTREAM_TASKS]])


In [ ]:
figsize = (5.2, 3.6)
fig, ax = plt.subplots(figsize=figsize, constrained_layout=True)

scatter_groups(ax, format_df, "format", format_styles, "size", "overall", label=True)
plot_bf16(ax, baseline["overall"])
plot_s3d8(ax, our_size, our_perf)

ax.annotate(
    LABELS["s3d8"],
    xy=(our_size, our_perf),
    xytext=(10, -5),
    textcoords="offset points",
    fontsize=10,
    ha="left",
    va="bottom",
)
ax.set(
    # title="Format Comparison",
    xlabel="Model size [MB]",
    ylabel="Average task performance",
)
finish_size_axis(ax, size_to_bpp)

handles, _ = ax.get_legend_handles_labels()
ax.legend(
    handles=[*handles, s3d8_handle(), model_legend_handles(compact=True)[-1]],
    loc="lower right",
    frameon=True,
    fancybox=False,
    edgecolor="0.8",
    labelspacing=0.35,
    handletextpad=0.5,
)

save_figure(fig, "fig_format_overall.pdf")
plt.show()


In [ ]:
figsize = (6.8, 5.6)
fig, axes = plt.subplots(2, 2, figsize=figsize, sharex=True, constrained_layout=True)

for ax, task in zip(axes.flatten(), DOWNSTREAM_TASKS):
    scatter_groups(ax, format_df, "format", format_styles, "size", task)
    plot_bf16(ax, baseline[task])
    plot_s3d8(ax, our_size, our_task_scores[task])

    ax.set(title=TASK_TITLES[task], xlabel="Model size [MB]", ylabel=get_metric(task))
    finish_size_axis(ax, size_to_bpp)

fig.legend(
    handles=[*style_handles(format_styles), s3d8_handle(), model_legend_handles(compact=True)[-1]],
    loc="lower center",
    ncol=5,
    mode="expand",
    frameon=False,
    bbox_to_anchor=(0.0, -0.09, 1.0, 0.08),
    labelspacing=0.35,
    handletextpad=0.5,
)

save_figure(fig, "fig_format_tasks.pdf")
plt.show()


In [ ]:
loss_specs = [("train_loss", "Training loss"), ("val_loss", "Validation loss")]
figsize = (6.8, 3.0)
fig, axes = plt.subplots(1, 2, figsize=figsize, constrained_layout=True)

for ax, (key, title) in zip(axes, loss_specs):
    scatter_groups(ax, format_df.dropna(subset=[key]), "format", format_styles, "size", key)
    plot_s3d8(ax, our_size, our_loss_scores[key])

    ax.set(title=title, xlabel="Model size [MB]", ylabel="KL Loss")
    finish_size_axis(ax, size_to_bpp, y_margin=0.10)

fig.legend(
    handles=[*style_handles(format_styles), s3d8_handle()],
    loc="lower center",
    ncol=4,
    frameon=False,
    bbox_to_anchor=(0.5, -0.08),
    labelspacing=0.35,
    handletextpad=0.5,
)

# save_figure(fig, "fig_format_loss.pdf")
plt.show()


In [ ]:
summary_columns = [
    "format",
    # "name",
    "size",
    "bits_per_param",
    "overall",
    *DOWNSTREAM_TASKS,
    # "train_loss",
    # "val_loss",
]
closest_format_df = (
    format_df.assign(size_delta=(format_df["size"] - our_size).abs())
    .sort_values(["format", "size_delta", "overall"], ascending=[True, True, False])
    .groupby("format", as_index=False)
    .head(1)
)
s3d8_summary = best_run.to_frame().T.assign(format="S3D8")
bf16_summary = pd.DataFrame(
    [
        {
            "format": "bfloat16",
            "name": bf16_run.name,
            "size": baseline_size,
            "bits_per_param": bf16_run.config["n_bits"]
            / get_num_params(bf16_run.config["model_name"]),
            "overall": baseline["overall"],
            **{task: baseline[task] for task in DOWNSTREAM_TASKS},
            "train_loss": None,
            "val_loss": bf16_run.summary.get("val/loss"),
        }
    ]
)

format_summary_df = pd.concat(
    [s3d8_summary, bf16_summary, closest_format_df],
    ignore_index=True,
).loc[:, summary_columns]

format_summary_df = format_summary_df.sort_values(
    "format",
    key=lambda s: s.map(
        {"S3D8": 1, "bfloat16": 0, "INT": 2, "student-t": 3, "lloyd-max": 4}
    ),
)
format_summary_display_df = format_summary_df.copy()
format_summary_display_df["size"] = pd.to_numeric(format_summary_display_df["size"]).round().astype(int)
format_summary_display_df["bits_per_param"] = pd.to_numeric(
    format_summary_display_df["bits_per_param"]
).round(2)
metric_columns = ["overall", *DOWNSTREAM_TASKS]
format_summary_display_df[metric_columns] = format_summary_display_df[metric_columns].apply(
    pd.to_numeric
).round(3)
format_summary_display_df["format"] = format_summary_display_df["format"].replace(
    FORMAT_LATEX_LABELS
)
display(format_summary_display_df)


In [ ]:
paper_table_df = format_summary_display_df.rename(
    columns={
        "format": "Format",
        "size": "Size [MB]",
        "bits_per_param": "Bits",
        "overall": "Avg.",
        "vqa": "VQA",
        "chartqa": "ChartQA",
        "docvqa": "DocVQA",
        "ai2d": "AI2D",
    }
)

print(
    paper_table_df.to_latex(
        index=False,
        escape=False,
        column_format="lrrrrrrr",
    )
)


### Loss and Task Performance Correlation


Compare train and validation loss against average downstream task performance for the 2048-step format comparison sweep points.


In [ ]:
figsize = (5.2, 3.6)
fig, ax = plt.subplots(figsize=figsize, constrained_layout=True)

loss_perf_df = format_df.dropna(subset=["train_loss", "overall"])
scatter_groups(
    ax,
    loss_perf_df,
    "format",
    format_styles,
    "train_loss",
    "overall",
    label=True,
)
plot_s3d8(ax, our_loss_scores["train_loss"], our_perf)
ax.set_xscale("log")
ax.set(
    xlabel="Training loss",
    ylabel="Average task performance",
)
finish_axis(ax, x_margin=0.08, y_margin=0.08)
handles, _ = ax.get_legend_handles_labels()
ax.legend(
    handles=[*handles, s3d8_handle()],
    title="Format",
    loc="lower left",
    frameon=True,
    fancybox=False,
    edgecolor="0.8",
    labelspacing=0.35,
    handletextpad=0.5,
)

save_figure(fig, "fig_task_vs_train_loss.pdf")
plt.show()


In [ ]:
figsize = (5.2, 3.6)
fig, ax = plt.subplots(figsize=figsize, constrained_layout=True)

loss_perf_df = format_df.dropna(subset=["val_loss", "overall"])
scatter_groups(
    ax,
    loss_perf_df,
    "format",
    format_styles,
    "val_loss",
    "overall",
    label=True,
)
plot_s3d8(ax, our_loss_scores["val_loss"], our_perf)
ax.set_xscale("log")
ax.set(
    xlabel="Validation loss",
    ylabel="Average task performance",
)
finish_axis(ax, x_margin=0.08, y_margin=0.08)
handles, _ = ax.get_legend_handles_labels()
ax.legend(
    handles=[*handles, s3d8_handle()],
    title="Format",
    loc="lower left",
    frameon=True,
    fancybox=False,
    edgecolor="0.8",
    labelspacing=0.35,
    handletextpad=0.5,
)

# save_figure(fig, "fig_task_vs_val_loss.pdf")
plt.show()


Spearman rank correlations for the same plotted points, including $\mathtt{S3D8}$.


In [ ]:
def spearman_corr(df: pd.DataFrame, x: str, y: str) -> float:
    corr_df = df.dropna(subset=[x, y])
    return corr_df[x].rank().corr(corr_df[y].rank())


loss_perf_with_s3d8_df = pd.concat(
    [
        format_df[["format", "train_loss", "val_loss", "overall"]],
        pd.DataFrame(
            [
                {
                    "format": "S3D8",
                    "train_loss": our_loss_scores["train_loss"],
                    "val_loss": our_loss_scores["val_loss"],
                    "overall": our_perf,
                }
            ]
        ),
    ],
    ignore_index=True,
)

spearman_df = pd.DataFrame(
    [
        {
            "x": "Training loss",
            "y": "Average task performance",
            "n": len(loss_perf_with_s3d8_df.dropna(subset=["train_loss", "overall"])),
            "spearman_rho": spearman_corr(
                loss_perf_with_s3d8_df, "train_loss", "overall"
            ),
        },
        {
            "x": "Validation loss",
            "y": "Average task performance",
            "n": len(loss_perf_with_s3d8_df.dropna(subset=["val_loss", "overall"])),
            "spearman_rho": spearman_corr(loss_perf_with_s3d8_df, "val_loss", "overall"),
        },
    ]
)

display(spearman_df.assign(spearman_rho=spearman_df["spearman_rho"].round(3)))


### Abblation: keeping layers in higher precision


In [ ]:
def higher_precision_layers(overrides: dict[str, Any] | None) -> str:
    keys = set((overrides or {}).keys())
    has_output = any(key.startswith("language_model.lm_head") for key in keys)
    has_vision = any(
        key.startswith("vision_model") or key.startswith("multi_modal_projector")
        for key in keys
    )

    if has_output and has_vision:
        return "LLM output + vision encoder"
    if has_output:
        return "LLM output"
    if has_vision:
        return "Vision encoder"
    return "None"


def cleanup_ablation_run(run: Run) -> dict[str, Any]:
    config = run.config
    scores = {
        task: run.summary[f"downstream.{task}.{get_metric(task)}"]
        for task in DOWNSTREAM_TASKS
    }
    history = run.history()

    return {
        "higher_precision_layers": higher_precision_layers(
            config["quantisation"].get("overrides")
        ),
        "n_steps": config["training"]["n_steps"],
        "rotation": config["training"].get("rotate_text_residual"),
        "activation_fmt": activation_fmt_name(config["quantisation"].get("activation_fmt")),
        "size": config["n_bits"] / (8 * 10**6),
        "bits_per_param": config["n_bits"] / get_num_params(config["model_name"]),
        "overall": overall_metric(list(scores.values())),
        "train_loss": history["train/loss"].tail(10).mean()
        if "train/loss" in history
        else None,
        **scores,
    }


ablation_order = [
    "None",
    "LLM output",
    "Vision encoder",
    "LLM output + vision encoder",
]
show_task_columns = True
performance_columns = ["overall", *(DOWNSTREAM_TASKS if show_task_columns else [])]
ablation_columns = [
    "higher_precision_layers",
    "size",
    "bits_per_param",
    *performance_columns,
    "train_loss",
]

ablation_df = pd.DataFrame(
    [cleanup_ablation_run(run) for run in load_runs(api, s3d8_run)]
)
ablation_df = ablation_df.loc[
    (ablation_df["n_steps"] == 2048)
    & (ablation_df["rotation"] == False)
    & (ablation_df["activation_fmt"] == "int")
    & ablation_df["higher_precision_layers"].isin(ablation_order)
].copy()
ablation_df["higher_precision_layers"] = pd.Categorical(
    ablation_df["higher_precision_layers"], categories=ablation_order, ordered=True
)
ablation_df = ablation_df.sort_values("higher_precision_layers")
if len(ablation_df) != 4:
    raise ValueError(f"Expected 4 ablation rows, found {len(ablation_df)}")

ablation_display_df = ablation_df.loc[:, ablation_columns].rename(
    columns={
        "higher_precision_layers": "INT8 layer",
        "size": "Size [MB]",
        "bits_per_param": "Bits",
        "overall": "Avg.",
        "vqa": "VQAv2",
        "chartqa": "ChartQA",
        "docvqa": "DocVQA",
        "ai2d": "AI2D",
        "train_loss": "Train loss",
    }
)
metric_columns = ["Avg.", *(TASK_TITLES[task] for task in DOWNSTREAM_TASKS if show_task_columns), "Train loss"]
ablation_display_df["Size [MB]"] = (
    pd.to_numeric(ablation_display_df["Size [MB]"]).round().astype(int)
)
ablation_display_df["Bits"] = pd.to_numeric(ablation_display_df["Bits"]).round(2)
ablation_display_df[metric_columns] = ablation_display_df[metric_columns].apply(
    pd.to_numeric
).round(3)
latex_formatters = {
    "Size [MB]": "{:.0f}".format,
    "Bits": "{:.2f}".format,
    **{column: "{:.3f}".format for column in metric_columns},
}

display(ablation_display_df)
print(
    ablation_display_df.to_latex(
        index=False,
        escape=False,
        formatters=latex_formatters,
        column_format="l" + "r" * (len(ablation_display_df.columns) - 1),
    )
)


### Generation Comparisons

Run the setup cell once to download/load artifacts and prepare 0-based comparison indices. Then rerun only the task-specific display cells while changing the example indices.

In [ ]:
from IPython.display import Markdown, display

from eval.vqa import TASKS as VQA_TASKS
from utility import get_task_outputs

# These default to the runs used above. Change them here to compare different runs.
bf16_generation_run_name = bf16_run_name
format_generation_run_name = best_s3d8_run


def get_finished_run_by_display_name(run_name: str) -> Run:
    runs = api.runs(PROJECT, filters={"display_name": run_name, "state": "finished"})
    if len(runs) != 1:
        raise ValueError(
            f"Expected exactly one finished run named {run_name!r}, found {len(runs)}"
        )
    return runs[0]


def records_by_id(records: list[dict[str, Any]]) -> dict[str, dict[str, Any]]:
    return {str(record["id"]): record for record in records}


def task_n_examples(run: Run, task: str) -> int | None:
    evaluation_config = get_field(run.config, "evaluation") or {}
    for task_config in get_field(evaluation_config, "tasks") or []:
        if get_field(task_config, "name") == task:
            return get_field(task_config, "n_examples")
    return None


def comparison_data_limit(task: str) -> int | None:
    limits = [
        task_n_examples(bf16_generation_run, task),
        task_n_examples(format_generation_run, task),
    ]
    limits = [limit for limit in limits if limit is not None]
    return max(limits) if limits else max(len(bf16_outputs[task]), len(format_outputs[task]))


def task_example_id(task: str, example: dict[str, Any]) -> Any:
    if task == "vqa":
        return example["question_id"]
    if task == "docvqa":
        return example["questionId"]
    return example["id"]


@cache
def task_examples_by_id(task: str, limit: int | None) -> dict[str, dict[str, Any]]:
    data = VQA_TASKS[task].data(limit=limit)
    return {str(task_example_id(task, example)): example for example in data}


def example_question_text(task: str, example: dict[str, Any]) -> str:
    if task == "chartqa":
        return example["query"]

    question = example["question"]
    if task == "ai2d":
        question += "\n\nOptions:\n" + VQA_TASKS["ai2d"]._format_options(example["options"])
    return question


def generation_text(record: dict[str, Any]) -> str:
    return str(record.get("output", "")).replace("<|eot_id|>", "").strip()


def metric_summary(record: dict[str, Any]) -> str:
    parts = []
    for key in ["accuracy", "accuracy_relaxed", "anls"]:
        if key in record:
            value = record[key]
            parts.append(f"{key}={value:.3g}" if isinstance(value, float) else f"{key}={value}")
    return ", ".join(parts) if parts else "no metrics"


def build_task_comparisons(task: str) -> list[dict[str, Any]]:
    format_task_by_id = records_by_id(format_outputs[task])
    examples = []
    for bf16_record in bf16_outputs[task]:
        key = str(bf16_record["id"])
        format_record = format_task_by_id.get(key)
        if format_record is None:
            continue
        examples.append(
            {
                "index": len(examples),
                "id": key,
                "bf16": bf16_record,
                "format": format_record,
            }
        )
    return examples


def display_source_example(task: str, example: dict[str, Any], example_id: str) -> None:
    if example is None:
        print("Question/image: not found in the loaded task data")
        return

    question = example_question_text(task, example)
    display(Markdown(f"**Question ID:** `{example_id}`\n\n**Question:**\n\n{question}"))
    image = example["image"].copy()
    image.thumbnail(globals().get("max_display_image_size", (700, 500)))
    display(image)


def print_generation_comparison(entry: dict[str, Any]) -> None:
    bf16_record = entry["bf16"]
    format_record = entry["format"]
    answers = bf16_record.get("answers", format_record.get("answers"))

    print(f"Answers: {answers}")
    print(f"\n{bf16_generation_run_name} / bfloat16 ({metric_summary(bf16_record)}):")
    print(generation_text(bf16_record))
    print(f"\n{format_generation_run_name} / format ({metric_summary(format_record)}):")
    print(generation_text(format_record))
    print("-" * 100)


def show_generation_comparison(
    task: str,
    example_index: int,
    *,
    show_questions: bool | None = None,
) -> None:
    examples = comparison_examples_by_task[task]
    if not 0 <= example_index < len(examples):
        raise IndexError(
            f"{task} example_index must be between 0 and {len(examples) - 1}, "
            f"got {example_index}"
        )

    show_questions = (
        globals().get("show_questions_and_images", False)
        if show_questions is None
        else show_questions
    )
    entry = examples[example_index]
    print("\n" + "=" * 100)
    print(f"{TASK_TITLES[task]} ({task})")
    print(f"Example index: {example_index} / {len(examples) - 1}")
    print(f"Question ID: {entry['id']}")

    if show_questions:
        source_examples = task_examples_by_id(task, comparison_data_limit(task))
        display_source_example(task, source_examples.get(entry["id"]), entry["id"])

    print_generation_comparison(entry)


bf16_generation_run = get_finished_run_by_display_name(bf16_generation_run_name)
format_generation_run = get_finished_run_by_display_name(format_generation_run_name)

bf16_outputs = get_task_outputs(bf16_generation_run)
format_outputs = get_task_outputs(format_generation_run)

comparison_examples_by_task = {
    task: build_task_comparisons(task) for task in DOWNSTREAM_TASKS
}
comparison_summary_df = pd.DataFrame(
    [
        {
            "task": task,
            "n_shared_examples": len(examples),
            "valid_index_range": f"0-{len(examples) - 1}",
        }
        for task, examples in comparison_examples_by_task.items()
    ]
)
display(comparison_summary_df)

#### Display Controls

In [ ]:
# Display controls. Rerun this cell and any task display cell after changing these.
show_questions_and_images = False
max_display_image_size = (700, 500)

#### VQAv2

In [ ]:
# Choose a 0-based example index from comparison_summary_df.
vqa_example_index = 0
show_generation_comparison("vqa", vqa_example_index)

#### ChartQA

In [ ]:
# Choose a 0-based example index from comparison_summary_df.
chartqa_example_index = 1
show_generation_comparison("chartqa", chartqa_example_index)

#### DocVQA

In [ ]:
# Choose a 0-based example index from comparison_summary_df.
docvqa_example_index = 36
show_generation_comparison("docvqa", docvqa_example_index)

#### AI2D

In [ ]:
# Choose a 0-based example index from comparison_summary_df.
ai2d_example_index = 0
show_generation_comparison("ai2d", ai2d_example_index)